# 07 — Random Survival Forest — FINAL

Khác bản cũ:

- Không hard-code 50,000 train rows.
- Không hard-code 50 trees.
- Không chỉ đánh giá 5,000 validation/test issues.
- Dùng cấu hình FINAL trong `experiment.yaml`.
- Harrell/IPCW/AUC chạy trên toàn bộ held-out split.
- IBS dùng sample deterministic, memory-safe.
- Chạy cả Day-0 và Day-7.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys, yaml, json, time, gc, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

SEED = int(CFG["random_seed"])
MODES = list(CFG.get("evaluation", {}).get("modes", ["day0"]))

print("ROOT =", ROOT)
print("random_seed =", SEED)
print("modes =", MODES)

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42
modes = ['day0', 'day7']


In [3]:
from src.features import feature_spec, sample_training_rows
from src.models import fit_rsf, save_artifact
from src.evaluation import evaluate_survival_model
from src.splitting import make_project_split

model_dir = ROOT / "results" / "models"
table_dir = ROOT / "results" / "tables"
model_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

all_rsf_metrics = []

In [4]:
from sklearn.model_selection import train_test_split


def make_metric_sample(
    df,
    max_rows,
    random_state,
):
    """
    Tạo evaluation sample reproducible.

    Phân tầng theo:
        repository + project + event

    nhằm giữ gần đúng:
        - tỷ lệ project
        - tỷ lệ resolved/censored

    trong tập evaluation.
    """

    if max_rows is None or len(df) <= int(max_rows):
        return df.copy()

    max_rows = int(max_rows)

    sample_df = df.copy()

    strata = (
        sample_df["repository"].astype(str)
        + "::"
        + sample_df["project"].astype(str)
        + "::event="
        + sample_df["event"].astype(str)
    )

    # Nếu có strata quá hiếm thì fallback sang random sample.
    counts = strata.value_counts()

    if (counts < 2).any():
        return sample_df.sample(
            n=max_rows,
            random_state=random_state,
        ).copy()

    selected, _ = train_test_split(
        sample_df,
        train_size=max_rows,
        random_state=random_state,
        stratify=strata,
    )

    return selected.copy()


# ============================================================
# RSF FINAL
# ============================================================

all_rsf_metrics = []

for mode in MODES:

    print("\n" + "=" * 100)
    print("RSF FINAL MODE:", mode)
    print("=" * 100)

    # --------------------------------------------------------
    # 1. Load dataset
    # --------------------------------------------------------

    data_path = (
        ROOT
        / "data"
        / "processed"
        / f"model_{mode}.parquet"
    )

    if not data_path.exists():
        raise FileNotFoundError(
            f"Không thấy {data_path}. "
            "Hãy chạy lại 04_feature_engineering.ipynb."
        )

    data = pd.read_parquet(
        data_path
    )


    # --------------------------------------------------------
    # 2. Project-disjoint split
    # --------------------------------------------------------

    train, val, test = make_project_split(
        data,
        random_state=SEED,
    )

    print(
        "Train projects:",
        sorted(
            train[
                "project_uid"
            ].unique()
        ),
    )

    print(
        "Validation projects:",
        sorted(
            val[
                "project_uid"
            ].unique()
        ),
    )

    print(
        "Test projects:",
        sorted(
            test[
                "project_uid"
            ].unique()
        ),
    )


    # --------------------------------------------------------
    # 3. Feature definition
    # --------------------------------------------------------

    strict = bool(
        CFG["features"][
            "strict_no_leakage"
        ]
    )

    landmark = int(
        CFG["features"][
            "landmark_days"
        ]
    )

    numeric_cols, categorical_cols = feature_spec(
        mode,
        strict_no_leakage=strict,
        landmark_days=landmark,
    )


    # --------------------------------------------------------
    # 4. Training sample
    # --------------------------------------------------------

    max_train_rows = (
        CFG["models"]
        .get("max_train_rows")
    )

    train_fit = sample_training_rows(
        train,
        max_rows=max_train_rows,
        random_state=SEED,
    )

    print(
        "Train rows used:",
        len(train_fit),
        "/",
        len(train),
    )


    # --------------------------------------------------------
    # 5. Evaluation samples
    # --------------------------------------------------------

    eval_cfg = CFG[
        "evaluation"
    ]

    max_metric_rows = (
        eval_cfg.get(
            "max_metric_rows",
            10000,
        )
    )

    val_eval = make_metric_sample(
        val,
        max_rows=max_metric_rows,
        random_state=SEED + 100,
    )

    test_eval = make_metric_sample(
        test,
        max_rows=max_metric_rows,
        random_state=SEED + 200,
    )

    print(
        "Validation rows used:",
        len(val_eval),
        "/",
        len(val),
    )

    print(
        "Test rows used:",
        len(test_eval),
        "/",
        len(test),
    )

    print(
        "Validation events:",
        int(
            val_eval["event"].sum()
        ),
        "/",
        len(val_eval),
    )

    print(
        "Test events:",
        int(
            test_eval["event"].sum()
        ),
        "/",
        len(test_eval),
    )


    # --------------------------------------------------------
    # 6. RSF config
    # --------------------------------------------------------

    rsf_cfg = (
        CFG["models"][
            "rsf"
        ].copy()
    )

    rsf_cfg[
        "random_state"
    ] = SEED

    print(
        "RSF config:",
        rsf_cfg,
    )


    # --------------------------------------------------------
    # 7. Train RSF
    # --------------------------------------------------------

    print(
        "\n[1/3] Training RSF..."
    )

    fit_started = time.time()

    rsf_artifact = fit_rsf(
        train_df=train_fit,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        **rsf_cfg,
    )

    fit_seconds = (
        time.time()
        - fit_started
    )

    print(
        f"RSF training finished "
        f"in {fit_seconds / 60:.2f} minutes"
    )


    # --------------------------------------------------------
    # 8. Save model NGAY sau khi train
    # --------------------------------------------------------
    #
    # Nếu evaluation lỗi thì không phải train lại model.
    # --------------------------------------------------------

    model_path = (
        model_dir
        / f"rsf_{mode}.joblib"
    )

    save_artifact(
        rsf_artifact,
        model_path,
    )

    print(
        "Saved model:",
        model_path,
    )


    # --------------------------------------------------------
    # 9. Validation evaluation
    # --------------------------------------------------------

    print(
        "\n[2/3] Evaluating validation..."
    )

    val_started = time.time()

    metrics_val = evaluate_survival_model(
        rsf_artifact,
        train_df=train_fit,
        test_df=val_eval,

        horizons_days=(
            eval_cfg[
                "horizons_days"
            ]
        ),

        max_ibs_rows=(
            eval_cfg.get(
                "max_ibs_rows",
                2000,
            )
        ),

        survival_batch_size=(
            eval_cfg.get(
                "survival_batch_size",
                16,
            )
        ),

        risk_batch_size=(
            eval_cfg.get(
                "risk_batch_size",
                256,
            )
        ),

        random_state=SEED,
    )

    val_seconds = (
        time.time()
        - val_started
    )

    print(
        f"Validation finished "
        f"in {val_seconds / 60:.2f} minutes"
    )

    print(
        "Validation metrics:"
    )

    print(
        metrics_val
    )


    # --------------------------------------------------------
    # 10. Test evaluation
    # --------------------------------------------------------

    print(
        "\n[3/3] Evaluating test..."
    )

    test_started = time.time()

    metrics_test = evaluate_survival_model(
        rsf_artifact,
        train_df=train_fit,
        test_df=test_eval,

        horizons_days=(
            eval_cfg[
                "horizons_days"
            ]
        ),

        max_ibs_rows=(
            eval_cfg.get(
                "max_ibs_rows",
                2000,
            )
        ),

        survival_batch_size=(
            eval_cfg.get(
                "survival_batch_size",
                16,
            )
        ),

        risk_batch_size=(
            eval_cfg.get(
                "risk_batch_size",
                256,
            )
        ),

        random_state=SEED + 1,
    )

    test_seconds = (
        time.time()
        - test_started
    )

    print(
        f"Test finished "
        f"in {test_seconds / 60:.2f} minutes"
    )

    print(
        "Test metrics:"
    )

    print(
        metrics_test
    )


    # --------------------------------------------------------
    # 11. Total runtime
    # --------------------------------------------------------

    total_seconds = (
        fit_seconds
        + val_seconds
        + test_seconds
    )

    print(
        "\nTotal RSF runtime:",
        f"{total_seconds / 60:.2f} minutes",
    )


    # --------------------------------------------------------
    # 12. Save evaluation sample IDs
    # --------------------------------------------------------
    #
    # Rất quan trọng cho reproducibility.
    # --------------------------------------------------------

    eval_keys = []

    for split_name, eval_df in [
        (
            "validation",
            val_eval,
        ),
        (
            "test",
            test_eval,
        ),
    ]:

        temp = eval_df[
            [
                "repository",
                "project",
                "issue_key",
            ]
        ].copy()

        temp[
            "mode"
        ] = mode

        temp[
            "split"
        ] = split_name

        eval_keys.append(
            temp
        )

    eval_keys_df = pd.concat(
        eval_keys,
        ignore_index=True,
    )

    eval_keys_df.to_csv(
        table_dir
        / f"rsf_eval_sample_{mode}.csv",
        index=False,
    )


    # --------------------------------------------------------
    # 13. Build metrics dataframe
    # --------------------------------------------------------

    metrics_df = pd.DataFrame(
        [
            {
                "model": "RSF",
                "mode": mode,
                "split": "validation",

                "train_rows": len(
                    train_fit
                ),

                "full_split_rows": len(
                    val
                ),

                "metric_rows": len(
                    val_eval
                ),

                "fit_seconds": (
                    fit_seconds
                ),

                "eval_seconds": (
                    val_seconds
                ),

                **metrics_val,
            },

            {
                "model": "RSF",
                "mode": mode,
                "split": "test",

                "train_rows": len(
                    train_fit
                ),

                "full_split_rows": len(
                    test
                ),

                "metric_rows": len(
                    test_eval
                ),

                "fit_seconds": (
                    fit_seconds
                ),

                "eval_seconds": (
                    test_seconds
                ),

                **metrics_test,
            },
        ]
    )


    # --------------------------------------------------------
    # 14. Save RSF metrics
    # --------------------------------------------------------

    metrics_path = (
        table_dir
        / f"rsf_metrics_{mode}.csv"
    )

    metrics_df.to_csv(
        metrics_path,
        index=False,
    )

    all_rsf_metrics.append(
        metrics_df
    )

    print(
        "\nSaved metrics:",
        metrics_path,
    )

    display(
        metrics_df
    )


    # --------------------------------------------------------
    # 15. Cox comparison
    # --------------------------------------------------------
    #
    # Chỉ tạo bảng comparison tạm.
    #
    # LƯU Ý:
    # Nếu Cox cũ được evaluate trên full split,
    # thì chưa phải apples-to-apples với RSF sample 10k.
    # File 06 nên được cập nhật dùng cùng metric sampling.
    # --------------------------------------------------------

    cox_file = (
        table_dir
        / f"cox_metrics_{mode}.csv"
    )

    if cox_file.exists():

        cox_df = pd.read_csv(
            cox_file
        )

        # Chỉ gộp nếu Cox cũng có thông tin metric_rows.
        if "metric_rows" in cox_df.columns:

            comparison = pd.concat(
                [
                    cox_df,
                    metrics_df,
                ],
                ignore_index=True,
            )

            comparison.to_csv(
                table_dir
                / f"model_comparison_{mode}.csv",
                index=False,
            )

            print(
                "Saved Cox/RSF comparison."
            )

        else:

            print(
                "\nWARNING:"
            )

            print(
                "Cox metrics hiện tại chưa dùng "
                "cùng evaluation sampling."
            )

            print(
                "Không tạo model_comparison mới "
                "để tránh so sánh không công bằng."
            )


    # --------------------------------------------------------
    # 16. Cleanup
    # --------------------------------------------------------

    del data
    del train
    del val
    del test

    del val_eval
    del test_eval

    del train_fit
    del rsf_artifact

    gc.collect()


# ============================================================
# Save all modes
# ============================================================

rsf_all = pd.concat(
    all_rsf_metrics,
    ignore_index=True,
)

rsf_all.to_csv(
    table_dir
    / "rsf_metrics_ALL_MODES.csv",
    index=False,
)

rsf_all


RSF FINAL MODE: day0
Train projects: ['Apache::HIVE', 'Jira::CONFSERVER', 'Jira::JRACLOUD', 'Jira::JRASERVER', 'MariaDB::MDEV', 'Mojang::MC', 'Mojang::MCPE', 'Sonatype::OSSRH']
Validation projects: ['Apache::FLEX', 'Sakai::SAK']
Test projects: ['MongoDB::SERVER', 'Qt::QTBUG']
Train rows used: 50000 / 600802
Validation rows used: 10000 / 78741
Test rows used: 10000 / 156100
Validation events: 8972 / 10000
Test events: 8359 / 10000
RSF config: {'n_estimators': 150, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 50, 'max_features': 'sqrt', 'n_jobs': 1, 'random_state': 42}

[1/3] Training RSF...
RSF training finished in 17.26 minutes
Saved model: d:\VNUK\Eureka 2026\Eureka_2026\results\models\rsf_day0.joblib

[2/3] Evaluating validation...
Validation finished in 20.78 minutes
Validation metrics:
{'n_test': 10000, 'events_test': 8972, 'harrell_c': 0.559023357591775, 'n_ipcw_eval': 10000, 'ipcw_c': 0.5634513109066819, 'auc_30': 0.5899526878932364, 'auc_60': 0.5852926342131664

,model,mode,split,train_rows,full_split_rows,metric_rows,fit_seconds,eval_seconds,n_test,events_test,...,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,RSF,day0,validation,50000,78741,10000,1035.405117,1246.674994,10000,8972,...,10000,0.563451,0.589953,0.585293,0.581583,0.588795,2000,0.175444,0.174827,-0.000617
1,RSF,day0,test,50000,156100,10000,1035.405117,1253.780467,10000,8359,...,10000,0.526920,0.533525,0.544286,0.551810,0.536546,2000,0.174295,0.179200,0.004905



Cox metrics hiện tại chưa dùng cùng evaluation sampling.
Không tạo model_comparison mới để tránh so sánh không công bằng.

RSF FINAL MODE: day7
Train projects: ['Apache::HIVE', 'Jira::CONFSERVER', 'Jira::JRACLOUD', 'Jira::JRASERVER', 'MariaDB::MDEV', 'Mojang::MC', 'Mojang::MCPE', 'Sonatype::OSSRH']
Validation projects: ['Apache::FLEX', 'Sakai::SAK']
Test projects: ['MongoDB::SERVER', 'Qt::QTBUG']
Train rows used: 50000 / 250679
Validation rows used: 10000 / 55295
Test rows used: 10000 / 114055
Validation events: 8540 / 10000
Test events: 7765 / 10000
RSF config: {'n_estimators': 150, 'max_depth': 8, 'min_samples_split': 100, 'min_samples_leaf': 50, 'max_features': 'sqrt', 'n_jobs': 1, 'random_state': 42}

[1/3] Training RSF...
RSF training finished in 34.07 minutes
Saved model: d:\VNUK\Eureka 2026\Eureka_2026\results\models\rsf_day7.joblib

[2/3] Evaluating validation...
Validation finished in 21.28 minutes
Validation metrics:
{'n_test': 10000, 'events_test': 8540, 'harrell_c': 0.5336

,model,mode,split,train_rows,full_split_rows,metric_rows,fit_seconds,eval_seconds,n_test,events_test,...,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,RSF,day7,validation,50000,55295,10000,2044.183421,1276.938651,10000,8540,...,10000,0.554187,0.542686,0.551810,0.553877,0.546306,2000,0.212788,0.209454,-0.003334
1,RSF,day7,test,50000,114055,10000,2044.183421,1331.802075,10000,7765,...,10000,0.599468,0.674013,0.662708,0.663484,0.669966,2000,0.191921,0.204754,0.012833



Cox metrics hiện tại chưa dùng cùng evaluation sampling.
Không tạo model_comparison mới để tránh so sánh không công bằng.


,model,mode,split,train_rows,full_split_rows,metric_rows,fit_seconds,eval_seconds,n_test,events_test,...,n_ipcw_eval,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs_rows,ibs,km_ibs,ibs_gain_vs_km
0,RSF,day0,validation,50000,78741,10000,1035.405117,1246.674994,10000,8972,...,10000,0.563451,0.589953,0.585293,0.581583,0.588795,2000,0.175444,0.174827,-0.000617
1,RSF,day0,test,50000,156100,10000,1035.405117,1253.780467,10000,8359,...,10000,0.526920,0.533525,0.544286,0.551810,0.536546,2000,0.174295,0.179200,0.004905
2,RSF,day7,validation,50000,55295,10000,2044.183421,1276.938651,10000,8540,...,10000,0.554187,0.542686,0.551810,0.553877,0.546306,2000,0.212788,0.209454,-0.003334
3,RSF,day7,test,50000,114055,10000,2044.183421,1331.802075,10000,7765,...,10000,0.599468,0.674013,0.662708,0.663484,0.669966,2000,0.191921,0.204754,0.012833


In [5]:
parts = []
for mode in MODES:
    p = table_dir / f"model_comparison_{mode}.csv"
    if p.exists():
        parts.append(pd.read_csv(p))

all_comparison = pd.concat(parts, ignore_index=True)
all_comparison.to_csv(
    table_dir / "model_comparison_ALL_MODES.csv",
    index=False,
)

cols = [
    "model", "mode", "split", "n_test",
    "harrell_c", "ipcw_c", "mean_dynamic_auc",
    "ibs", "km_ibs", "ibs_gain_vs_km",
]
display(all_comparison[[c for c in cols if c in all_comparison.columns]])

,model,mode,split,n_test,harrell_c,ipcw_c,mean_dynamic_auc,ibs
0,CoxPH,day0,validation,87261,0.595495,0.588819,0.636529,0.198562
1,CoxPH,day0,test,172132,0.374037,0.378573,0.312770,NaN
2,RSF,day0,validation,5000,0.616681,0.606134,0.673044,0.195977
3,RSF,day0,test,5000,0.373740,0.377909,0.331437,0.165449
